In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

In [5]:
RAW_PATH   = Path("data/raw/online_retail_II.xlsx")
OUT_DIR    = Path("outputs/eda")
CAL_END    = "2011-03-01"
NON_PRODUCT_CODES = {"POST","DOT","M","AMAZONFEE","BANK CHARGES","C2","D","CRUK","S","PADS"}
 
OUT_DIR.mkdir(parents=True, exist_ok=True)
 
PALETTE = {
    "primary"    : "#2E4057",
    "secondary"  : "#048A81",
    "accent"     : "#E76F51",
    "light"      : "#8ECAE6",
    "gray"       : "#AAAAAA",
}
 
plt.rcParams.update({
    "figure.figsize"     : (10, 5),
    "font.family"        : "sans-serif",
    "font.size"          : 11,
    "axes.titlesize"     : 13,
    "axes.labelsize"     : 12,
    "axes.spines.top"    : False,
    "axes.spines.right"  : False,
    "axes.grid"          : True,
    "grid.alpha"         : 0.3,
    "grid.linestyle"     : "--",
})
 
def save(fig, name):
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    print(f"  ✓ saved {name}.png")
    plt.close(fig)
 
# ── 1. Load ───────────────────────────────────────────────────────────────────
print("\n=== 1. Loading raw data ===")
if not RAW_PATH.exists():
    print(f"\n⚠  Dataset not found at {RAW_PATH}")
    print("   Please upload 'online_retail_II.xlsx' and place it at data/raw/")
    exit(1)
 
df1 = pd.read_excel(RAW_PATH, sheet_name="Year 2009-2010")
df2 = pd.read_excel(RAW_PATH, sheet_name="Year 2010-2011")
raw = pd.concat([df1, df2], ignore_index=True)
raw.columns = raw.columns.str.strip()
print(f"  Rows: {len(raw):,}  |  Columns: {list(raw.columns)}")
print(f"  Date range: {raw['InvoiceDate'].min()} → {raw['InvoiceDate'].max()}")
print(raw.dtypes)
print(raw.isnull().sum().rename("nulls"))
 


=== 1. Loading raw data ===
  Rows: 1,067,371  |  Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
  Date range: 2009-12-01 07:45:00 → 2011-12-09 12:50:00
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
Name: nulls, dtype: int64


In [8]:
df2.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,541910.000000,541910,541910.000000,406830.000000
mean,9.552234,2011-07-04 13:35:22.342307,4.611138,15287.684160
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.080957,NaN,96.759765,1713.603074


In [9]:
print("\n=== 2. Cleaning ===")
n0 = len(raw)
df = raw.copy()
 
df = df.dropna(subset=["Customer ID"])
n_no_cid = n0 - len(df)
 
df["Invoice"]   = df["Invoice"].astype(str)
cancels         = df["Invoice"].str.startswith("C")
df              = df[~cancels]
n_cancel        = cancels.sum()
 
df["StockCode"] = df["StockCode"].astype(str).str.strip().str.upper()
non_prod        = df["StockCode"].isin(NON_PRODUCT_CODES)
df              = df[~non_prod]
n_nonprod       = non_prod.sum()
 
neg_mask        = ~((df["Quantity"] > 0) & (df["Price"] > 0))
df              = df[~neg_mask]
n_neg           = neg_mask.sum()
 
df["Revenue"]      = df["Quantity"] * df["Price"]
df["InvoiceDate"]  = pd.to_datetime(df["InvoiceDate"])
df["Customer ID"]  = df["Customer ID"].astype(int)
 
print(f"  Raw:              {n0:>10,}")
print(f"  - no Customer ID: {n_no_cid:>10,}  ({n_no_cid/n0:.1%})")
print(f"  - cancellations:  {n_cancel:>10,}  ({n_cancel/n0:.1%})")
print(f"  - non-product:    {n_nonprod:>10,}  ({n_nonprod/n0:.1%})")
print(f"  - neg qty/price:  {n_neg:>10,}  ({n_neg/n0:.1%})")
print(f"  Clean total:      {len(df):>10,}  ({len(df)/n0:.1%} retained)")
 
# Cleaning waterfall chart
fig, ax = plt.subplots(figsize=(9, 5))
labels  = ["Raw", "No Customer ID", "Cancellations", "Non-product", "Neg qty/price", "Clean"]
values  = [n0, n_no_cid, n_cancel, n_nonprod, n_neg, len(df)]
colors  = [PALETTE["primary"]] + [PALETTE["accent"]]*4 + [PALETTE["secondary"]]
bars    = ax.bar(labels, values, color=colors, edgecolor="white")
for bar, val in zip(bars, values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
            f"{val:,}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Row count")
ax.set_title("Data Cleaning Waterfall")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f"{x/1e3:.0f}K"))
save(fig, "01_cleaning_waterfall")


=== 2. Cleaning ===
  Raw:               1,067,371
  - no Customer ID:    243,007  (22.8%)
  - cancellations:      18,744  (1.8%)
  - non-product:         2,878  (0.3%)
  - neg qty/price:          63  (0.0%)
  Clean total:         802,679  (75.2% retained)
  ✓ saved 01_cleaning_waterfall.png


In [10]:
# ── 3. Transaction-level EDA ──────────────────────────────────────────────────
print("\n=== 3. Transaction-level EDA ===")
 
# Revenue per invoice
inv = df.groupby("Invoice")["Revenue"].sum()
print(f"  Unique invoices:  {inv.nunique():,}")
print(f"  Invoice value — median: £{inv.median():.2f}  mean: £{inv.mean():.2f}  p95: £{inv.quantile(.95):.2f}")
 
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(inv.clip(upper=inv.quantile(.99)), bins=60,
             color=PALETTE["primary"], edgecolor="white", alpha=0.85)
axes[0].set_title("Invoice Value Distribution (clipped at 99th pct)")
axes[0].set_xlabel("Revenue per Invoice (GBP)")
axes[0].set_ylabel("Count")
 
axes[1].hist(np.log1p(inv), bins=60,
             color=PALETTE["secondary"], edgecolor="white", alpha=0.85)
axes[1].set_title("Invoice Value — log1p scale")
axes[1].set_xlabel("log(1 + Revenue)")
save(fig, "02_invoice_value_dist")
 
# Quantity per line
fig, ax = plt.subplots()
ax.hist(df["Quantity"].clip(upper=df["Quantity"].quantile(.99)), bins=50,
        color=PALETTE["light"], edgecolor="white", alpha=0.85)
ax.set_title("Line Quantity Distribution (clipped at 99th pct)")
ax.set_xlabel("Quantity")
ax.set_ylabel("Line count")
save(fig, "03_quantity_dist")


=== 3. Transaction-level EDA ===
  Unique invoices:  26,784
  Invoice value — median: £304.50  mean: £475.97  p95: £1240.86
  ✓ saved 02_invoice_value_dist.png
  ✓ saved 03_quantity_dist.png


In [11]:
# ── 4. Temporal patterns ──────────────────────────────────────────────────────
print("\n=== 4. Temporal patterns ===")
 
df["YearMonth"] = df["InvoiceDate"].dt.to_period("M")
monthly_rev     = df.groupby("YearMonth")["Revenue"].sum()
monthly_cust    = df.groupby("YearMonth")["Customer ID"].nunique()
monthly_inv     = df.groupby("YearMonth")["Invoice"].nunique()
 
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
x = monthly_rev.index.astype(str)
 
axes[0].bar(x, monthly_rev.values, color=PALETTE["primary"], edgecolor="white")
axes[0].set_title("Monthly Revenue")
axes[0].set_ylabel("GBP")
axes[0].yaxis.set_major_formatter(ticker.FuncFormatter(lambda v,_: f"£{v/1e6:.1f}M"))
 
axes[1].bar(x, monthly_cust.values, color=PALETTE["secondary"], edgecolor="white")
axes[1].set_title("Monthly Unique Customers")
axes[1].set_ylabel("Customers")
 
axes[2].bar(x, monthly_inv.values, color=PALETTE["light"], edgecolor="white")
axes[2].set_title("Monthly Invoices")
axes[2].set_ylabel("Invoices")
 
for ax in axes:
    ax.tick_params(axis="x", rotation=45)
    cal_end_str = pd.Period(CAL_END, "M").strftime("%Y-%m")
    if cal_end_str in list(x):
        ax.axvline(list(x).index(cal_end_str)-0.5,
                   color=PALETTE["accent"], linestyle="--", linewidth=1.5,
                   label="Cal/Holdout split")
        ax.legend(fontsize=9)
 
save(fig, "04_monthly_timeseries")
 
# Day of week
df["DOW"] = df["InvoiceDate"].dt.day_name()
dow_order  = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
dow_rev    = df.groupby("DOW")["Revenue"].sum().reindex(dow_order)
 
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(dow_rev.index, dow_rev.values, color=PALETTE["primary"], edgecolor="white")
ax.set_title("Revenue by Day of Week")
ax.set_ylabel("Total Revenue (GBP)")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v,_: f"£{v/1e6:.1f}M"))
save(fig, "05_dow_revenue")


=== 4. Temporal patterns ===
  ✓ saved 04_monthly_timeseries.png
  ✓ saved 05_dow_revenue.png


In [14]:
print("\n=== 5. Customer-level RFM distributions ===")
 
cal_end_dt = pd.Timestamp(CAL_END)
cal = df[df["InvoiceDate"] < cal_end_dt].copy()
 
# Invoice-level
invoices = (
    cal.groupby(["Customer ID","Invoice"])
    .agg(invoice_date=("InvoiceDate","min"), invoice_revenue=("Revenue","sum"),
         country=("Country","first"))
    .reset_index()
)
cal_end_obs = cal["InvoiceDate"].max() + pd.Timedelta(days=1)
 
customers = []
for cid, grp in invoices.groupby("Customer ID"):
    grp = grp.sort_values("invoice_date")
    first, last = grp["invoice_date"].min(), grp["invoice_date"].max()
    n = len(grp)
    freq = n - 1
    recency = (last - first) / np.timedelta64(1, "W")
    T       = (cal_end_obs - first) / np.timedelta64(1, "W")
    mv      = grp.iloc[1:]["invoice_revenue"].mean() if freq > 0 else 0.0
    customers.append(dict(customer_id=cid, frequency=freq, recency=recency, T=T,
                          monetary_value=mv, n_purchases=n,
                          total_revenue=grp["invoice_revenue"].sum(),
                          country=grp["country"].mode().iloc[0]))
cust = pd.DataFrame(customers)
 
print(f"  Total customers (calibration): {len(cust):,}")
print(f"  Repeat purchasers:             {(cust.frequency>0).sum():,} ({(cust.frequency>0).mean():.1%})")
print(f"  One-time purchasers:           {(cust.frequency==0).sum():,} ({(cust.frequency==0).mean():.1%})")
print(f"  Mean frequency:  {cust.frequency.mean():.2f}")
print(f"  Median T (weeks):{cust['T'].median():.1f}")
print(f"  Mean monetary:   £{cust.loc[cust.frequency>0,'monetary_value'].mean():.2f}")
 
# Frequency distribution
fig, ax = plt.subplots()
ax.hist(cust["frequency"].clip(upper=cust["frequency"].quantile(.98)), bins=40,
        color=PALETTE["primary"], edgecolor="white", alpha=0.85)
ax.axvline(cust["frequency"].mean(), color=PALETTE["accent"], linestyle="--",
           linewidth=2, label=f"Mean: {cust['frequency'].mean():.1f}")
ax.set_title("Customer Frequency Distribution")
ax.set_xlabel("Repeat Purchases (frequency)")
ax.set_ylabel("Number of Customers")
ax.legend()
save(fig, "06_rfm_frequency")
 
# T distribution
fig, ax = plt.subplots()
ax.hist(cust["T"], bins=40, color=PALETTE["secondary"], edgecolor="white", alpha=0.85)
ax.set_title("Customer Observation Period T (weeks)")
ax.set_xlabel("T (weeks since first purchase)")
ax.set_ylabel("Customers")
save(fig, "07_rfm_T")
 
# Recency vs T scatter
fig, ax = plt.subplots()
ax.scatter(cust["T"], cust["recency"], alpha=0.15, s=8,
           color=PALETTE["primary"])
ax.plot([0, cust["T"].max()], [0, cust["T"].max()], "--",
        color=PALETTE["accent"], linewidth=1.5, label="recency = T (impossible)")
ax.set_xlabel("T (weeks)")
ax.set_ylabel("Recency (weeks)")
ax.set_title("Recency vs T — Customer Spread")
ax.legend()
save(fig, "08_recency_vs_T")
 
# Monetary value distribution (repeat customers only)
repeat = cust[cust.frequency > 0]
fig, ax = plt.subplots()
ax.hist(repeat["monetary_value"].clip(upper=repeat["monetary_value"].quantile(.98)),
        bins=40, color=PALETTE["secondary"], edgecolor="white", alpha=0.85)
ax.axvline(repeat["monetary_value"].mean(), color=PALETTE["accent"],
           linestyle="--", linewidth=2,
           label=f"Mean: £{repeat['monetary_value'].mean():.1f}")
ax.set_title("Monetary Value Distribution (Repeat Customers)")
ax.set_xlabel("Avg Transaction Value (GBP)")
ax.set_ylabel("Customers")
ax.legend()
save(fig, "09_rfm_monetary")
 
# RFM correlation heatmap
rfm_cols = ["frequency","recency","T","monetary_value"]
corr = cust[rfm_cols].corr()
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Blues", ax=ax,
            linewidths=0.5, square=True, cbar_kws={"shrink":.8})
ax.set_title("RFM Feature Correlations")
save(fig, "10_rfm_correlation")


=== 5. Customer-level RFM distributions ===
  Total customers (calibration): 4,522
  Repeat purchasers:             3,029 (67.0%)
  One-time purchasers:           1,493 (33.0%)
  Mean frequency:  3.78
  Median T (weeks):45.9
  Mean monetary:   £384.99
  ✓ saved 06_rfm_frequency.png
  ✓ saved 07_rfm_T.png
  ✓ saved 08_recency_vs_T.png
  ✓ saved 09_rfm_monetary.png
  ✓ saved 10_rfm_correlation.png


In [15]:
print("\n=== 6. Country analysis ===")
 
country_counts = cust["country"].value_counts()
top10 = country_counts.head(10)
print(top10.to_string())
 
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top10.index[::-1], top10.values[::-1],
               color=PALETTE["primary"], edgecolor="white")
for bar, val in zip(bars, top10.values[::-1]):
    ax.text(bar.get_width() + 15, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=9, color=PALETTE["gray"])
ax.set_xlabel("Number of Customers")
ax.set_title("Top 10 Countries by Customer Count")
save(fig, "11_country_customers")
 
# Revenue by country
rev_country = df.groupby("Country")["Revenue"].sum().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(rev_country.index[::-1], rev_country.values[::-1],
        color=PALETTE["secondary"], edgecolor="white")
ax.set_xlabel("Total Revenue (GBP)")
ax.set_title("Top 10 Countries by Revenue")
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda v,_: f"£{v/1e6:.1f}M"))
save(fig, "12_country_revenue")


=== 6. Country analysis ===
country
United Kingdom    4152
Germany             74
France              54
Spain               26
Netherlands         22
Belgium             19
Portugal            18
Sweden              16
Australia           15
Switzerland         14
  ✓ saved 11_country_customers.png
  ✓ saved 12_country_revenue.png


In [17]:
print("\n=== 7. Cal / Holdout split ===")
holdout = df[df["InvoiceDate"] >= cal_end_dt].copy()
 
print(f"  Calibration rows : {len(cal):,}  ({cal.InvoiceDate.min().date()} → {cal.InvoiceDate.max().date()})")
print(f"  Holdout rows     : {len(holdout):,}  ({holdout.InvoiceDate.min().date()} → {holdout.InvoiceDate.max().date()})")
print(f"  Cal customers    : {cal['Customer ID'].nunique():,}")
print(f"  Holdout customers: {holdout['Customer ID'].nunique():,}")
overlap = set(cal["Customer ID"].unique()) & set(holdout["Customer ID"].unique())
print(f"  Overlap (in both): {len(overlap):,} ({len(overlap)/cal['Customer ID'].nunique():.1%} of cal customers)")
 
# ── 8. Final summary ──────────────────────────────────────────────────────────
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
summary = {
    "Raw transactions"          : f"{len(raw):,}",
    "Clean transactions"        : f"{len(df):,}",
    "Unique customers (clean)"  : f"{df['Customer ID'].nunique():,}",
    "Unique products"           : f"{df['StockCode'].nunique():,}",
    "Unique countries"          : f"{df['Country'].nunique():,}",
    "Date range"                : f"{df.InvoiceDate.min().date()} → {df.InvoiceDate.max().date()}",
    "Total revenue"             : f"£{df.Revenue.sum():,.0f}",
    "Cal customers"             : f"{cust.customer_id.nunique():,}",
    "Repeat purchasers"         : f"{(cust.frequency>0).sum():,} ({(cust.frequency>0).mean():.1%})",
    "Mean frequency"            : f"{cust.frequency.mean():.2f}",
    "Mean T (weeks)"            : f"{cust['T'].mean():.1f}",
    "Mean monetary (repeat cust)": f"£{repeat.monetary_value.mean():.2f}",
}
for k, v in summary.items():
    print(f"  {k:<35} {v}")
 



=== 7. Cal / Holdout split ===
  Calibration rows : 473,386  (2009-12-01 → 2011-02-28)
  Holdout rows     : 329,293  (2011-03-01 → 2011-12-09)
  Cal customers    : 4,522
  Holdout customers: 4,009
  Overlap (in both): 2,670 (59.0% of cal customers)

SUMMARY
  Raw transactions                    1,067,371
  Clean transactions                  802,679
  Unique customers (clean)            5,861
  Unique products                     4,624
  Unique countries                    41
  Date range                          2009-12-01 → 2011-12-09
  Total revenue                       £17,438,960
  Cal customers                       4,522
  Repeat purchasers                   3,029 (67.0%)
  Mean frequency                      3.78
  Mean T (weeks)                      41.8
  Mean monetary (repeat cust)         £384.99
